# 🎵 Latin Music Genre Classifier — Interactive Demo

Classify Latin music into **Bachata · Merengue · Reggaeton · Salsa** using audio embeddings.

### How it works
1. **Upload** an audio file or **record** a ~30 s clip from your microphone
2. Choose a **model** and tweak the **confidence threshold**
3. Hit **Classify!** — results show the predicted genre plus per-genre probabilities

### Pipeline
```
Audio → CLAP embeddings (512-d) + BPM + 13 MFCCs
      → StandardScaler → Model → Genre probabilities
```

| Model | Description |
|---|---|
| **LightGBM** | Gradient boosted trees (fast, interpretable) |
| **XGBoost** | Gradient boosted trees (second opinion) |
| **NeuralNetwork** | Deep MLP with BatchNorm + Dropout |
| **Stacking** | LogReg meta-model trained on OOF base-model predictions |
| **WeightedVoting** | Optuna-tuned weighted average of all base models |

> **Dataset:** [Latin Music Embeddings](https://www.kaggle.com/datasets/g5fighter/latin-music-embeddings)  
> **Models:** trained in the companion [music-classifier](https://www.kaggle.com/code/g5fighter/music-classifier) notebook


## 1 · Install Dependencies

Run once per Kaggle session.


In [ ]:
!pip install -q gradio

## 2 · Imports


In [ ]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import librosa
import joblib
import gradio as gr

from transformers import ClapModel, ClapProcessor
from tensorflow import keras

print('✅ Imports OK')
print(f'   Gradio   : {gr.__version__}')
print(f'   PyTorch  : {torch.__version__}')
print(f'   librosa  : {librosa.__version__}')

## 3 · Configuration

The models are published at  
🔗 [g5fighter/latin-music-genre-classifiers](https://www.kaggle.com/models/g5fighter/latin-music-genre-classifiers)

**Add that model as an input** to this notebook (Add input → Models → search for it).

The full path to a file looks like this:
```
/kaggle/input/models/g5fighter/latin-music-genre-classifiers/tensorflow2/latin-music-genre-classifier/11/models/20260521_140358/manifest.json
```

The uploaded archive mirrors the training output exactly:
```
<mount root>/
└── models/
    └── 20260521_140358/      ← timestamped run folder (latest is picked automatically)
        ├── scaler.joblib
        ├── label_encoder.joblib
        ├── manifest.json
        ├── lightgbm_model.joblib
        ├── xgboost_model.joblib
        ├── neural_network.keras
        ├── stacking_meta.joblib
        ├── weighted_voting_weights.npy
        └── weighted_voting_model_order.joblib
```

Only change these constants if the model page shows different values:

| Constant | Current value |
|---|---|
| `MODEL_FRAMEWORK` | `tensorflow2` |
| `MODEL_VARIATION` | `latin-music-genre-classifier` |
| `MODEL_VERSION` | `11` |


In [ ]:
# ── Kaggle Models coordinates ─────────────────────────────────────────────────
MODEL_OWNER     = 'g5fighter'
MODEL_SLUG      = 'latin-music-genre-classifiers'
MODEL_FRAMEWORK = 'tensorflow2'                  # framework slug on the model page
MODEL_VARIATION = 'latin-music-genre-classifier' # variation name (singular)
MODEL_VERSION   = 11                             # V11 shown on the model page

# Full mount path:
#   /kaggle/input/models/g5fighter/latin-music-genre-classifiers/tensorflow2/latin-music-genre-classifier/11
_MOUNT = (
    f'/kaggle/input/models/{MODEL_OWNER}/{MODEL_SLUG}'
    f'/{MODEL_FRAMEWORK}/{MODEL_VARIATION}/{MODEL_VERSION}'
)

# Inside the mount the training output is nested under models/{run_id}/
# Pick the latest timestamped run automatically so bumping MODEL_VERSION is enough.
_runs_dir = os.path.join(_MOUNT, 'models')
_runs = sorted(
    d for d in os.listdir(_runs_dir)
    if os.path.isdir(os.path.join(_runs_dir, d)) and d[0].isdigit()
)
if not _runs:
    raise RuntimeError(f'No run folders found inside {_runs_dir}')

MODEL_PATH = os.path.join(_runs_dir, _runs[-1])   # .../models/20260521_140358
print(f'Using run : {_runs[-1]}')
print(f'Model path: {MODEL_PATH}')

# ── CLAP model (same one used during dataset generation) ──────────────────────
CLAP_MODEL_ID = 'laion/clap-htsat-unfused'

# ── Audio settings ────────────────────────────────────────────────────────────
FRAGMENT_DURATION_SEC = 30      # length of each analysis window (seconds)
MIN_FRAGMENT_DURATION = 5       # reject windows shorter than this (seconds)
CLAP_SR               = 48_000
LIBROSA_SR            = 22_050

# ── Feature column order (must match training) ────────────────────────────────
FEATURE_COLS = (
    ['bpm']
    + [f'mfcc_{i}' for i in range(13)]
    + [f'emb_{i}'  for i in range(512)]
)

print(f'Feature vector size: {len(FEATURE_COLS)}')

## 4 · Load CLAP Model

CLAP (`laion/clap-htsat-unfused`) is the same model used during dataset generation.  
If you have a `HF_TOKEN` Kaggle secret, logging in avoids rate-limiting.


In [ ]:
# Optional HF login (avoids rate-limiting for public models)
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(token=UserSecretsClient().get_secret('HF_TOKEN'), add_to_git_credential=False)
    print('✅ Logged in to Hugging Face')
except Exception:
    print('ℹ️  No HF_TOKEN found — continuing anonymously (may be rate-limited)')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

print(f'Loading CLAP model ({CLAP_MODEL_ID})…')
clap_processor = ClapProcessor.from_pretrained(CLAP_MODEL_ID)
clap_model     = ClapModel.from_pretrained(CLAP_MODEL_ID).to(device)
clap_model.eval()
print('✅ CLAP model ready')

## 5 · Load Trained Models

All artefacts live **flat** inside `MODEL_PATH` (no timestamp sub-folder — Kaggle Models
uses versioning instead):

| File | Contents |
|---|---|
| `scaler.joblib` | `StandardScaler` fitted on training fragments |
| `label_encoder.joblib` | `LabelEncoder` → genre names |
| `manifest.json` | Best pooling strategy + base-model order |
| `lightgbm_model.joblib` | Calibrated LightGBM |
| `xgboost_model.joblib` | Calibrated XGBoost |
| `neural_network.keras` | Keras MLP |
| `stacking_meta.joblib` | LogReg meta-model |
| `weighted_voting_weights.npy` | Optuna-tuned ensemble weights |
| `weighted_voting_model_order.joblib` | Model name list matching the weights |


In [ ]:
print(f'Loading models from: {MODEL_PATH}')

# ── Preprocessing ──────────────────────────────────────────────────────────────
scaler = joblib.load(f'{MODEL_PATH}/scaler.joblib')
le     = joblib.load(f'{MODEL_PATH}/label_encoder.joblib')
GENRES = le.classes_.tolist()
print(f'Genres: {GENRES}')

# ── Manifest (pooling strategy, base-model order) ──────────────────────────────
with open(f'{MODEL_PATH}/manifest.json') as fh:
    manifest = json.load(fh)
BEST_POOLING = manifest.get('best_pooling', 'max')
WV_ORDER     = manifest.get('model_order', ['LightGBM', 'XGBoost', 'NeuralNetwork'])
print(f'Best pooling from training : {BEST_POOLING}')
print(f'Base-model order           : {WV_ORDER}')

# ── Base models ────────────────────────────────────────────────────────────────
base_models = {
    'LightGBM'     : joblib.load(f'{MODEL_PATH}/lightgbm_model.joblib'),
    'XGBoost'      : joblib.load(f'{MODEL_PATH}/xgboost_model.joblib'),
    'NeuralNetwork': keras.models.load_model(f'{MODEL_PATH}/neural_network.keras'),
}

# ── Ensemble artefacts ─────────────────────────────────────────────────────────
stacking_meta = joblib.load(f'{MODEL_PATH}/stacking_meta.joblib')
wv_weights    = np.load(f'{MODEL_PATH}/weighted_voting_weights.npy')
wv_weights    = wv_weights / wv_weights.sum()   # guarantee they sum to 1

ALL_MODELS = list(base_models.keys()) + ['Stacking', 'WeightedVoting']

print('\n✅ All models loaded:')
for name in ALL_MODELS:
    print(f'   {name}')
print(f'\nWeighted-voting weights: { {n: round(w, 3) for n, w in zip(WV_ORDER, wv_weights)} }')

## 6 · Feature Extraction

Identical to the dataset-generator pipeline:
1. Convert audio to 48 kHz mono → CLAP embedding (512-d)
2. Resample to 22 050 Hz → BPM + 13 MFCCs
3. Concatenate into a 526-d feature vector

Long clips are split into consecutive 30 s windows; each window is one "fragment".


In [ ]:
def _to_mono_float(audio_np: np.ndarray) -> np.ndarray:
    """Convert any Gradio audio array to mono float32 in [-1, 1]."""
    if audio_np.ndim > 1:
        audio_np = audio_np.mean(axis=-1)
    audio_np = audio_np.astype(np.float32)
    if np.abs(audio_np).max() > 1.0:
        audio_np = audio_np / 32768.0
    return np.clip(audio_np, -1.0, 1.0)


def extract_fragment_features(frag_48k: np.ndarray) -> dict | None:
    """
    Compute features for a single 30 s fragment already at 48 kHz.
    Returns a feature dict or None if the fragment is invalid.
    """
    if len(frag_48k) < MIN_FRAGMENT_DURATION * CLAP_SR:
        return None
    if np.abs(frag_48k).max() < 1e-4:
        return None

    # ── CLAP embedding (512 dimensions) ──────────────────────────────────────
    inputs = clap_processor(
        audio=frag_48k, sampling_rate=CLAP_SR, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        actual = clap_model.module if hasattr(clap_model, 'module') else clap_model
        embedding = actual.get_audio_features(**inputs)[0].cpu().numpy().flatten()

    if np.isnan(embedding).any():
        return None

    # ── BPM + 13 MFCCs ───────────────────────────────────────────────────────
    y_22k = librosa.resample(frag_48k, orig_sr=CLAP_SR, target_sr=LIBROSA_SR)
    tempo, _ = librosa.beat.beat_track(y=y_22k, sr=LIBROSA_SR)
    mfcc_means = np.mean(
        librosa.feature.mfcc(y=y_22k, sr=LIBROSA_SR, n_mfcc=13), axis=1
    )

    record = {'bpm': float(np.mean(tempo))}
    for i, v in enumerate(mfcc_means):
        record[f'mfcc_{i}'] = float(v)
    for i, v in enumerate(embedding):
        record[f'emb_{i}'] = float(v)
    return record


def audio_to_features(
    audio_np: np.ndarray,
    sr: int,
    progress=None,
    progress_start: float = 0.05,
    progress_end: float = 0.80,
) -> tuple[np.ndarray | None, str | None]:
    """
    Convert a raw audio array to a scaled (n_fragments, n_features) matrix.
    Reports progress between progress_start and progress_end if a callable is given.
    Returns (X_scaled, error_message).
    """
    def _report(frac: float, desc: str):
        if progress is not None:
            value = progress_start + frac * (progress_end - progress_start)
            progress(value, desc=desc)

    _report(0.0, 'Resampling audio…')
    audio = _to_mono_float(audio_np)
    y_48k = librosa.resample(audio, orig_sr=sr, target_sr=CLAP_SR)

    # Split into consecutive 30 s windows
    window    = int(FRAGMENT_DURATION_SEC * CLAP_SR)
    fragments = [y_48k[s: s + window] for s in range(0, len(y_48k), window)]
    n         = len(fragments)

    records = []
    for i, frag in enumerate(fragments):
        _report((i + 0.5) / n, f'Extracting features — fragment {i + 1} / {n}…')
        feat = extract_fragment_features(frag)
        if feat is not None:
            records.append(feat)

    if not records:
        return None, (
            'No valid audio fragment found. '
            f'Please provide at least {MIN_FRAGMENT_DURATION} s of non-silent audio.'
        )

    _report(1.0, 'Scaling features…')
    df       = pd.DataFrame(records)[FEATURE_COLS]
    X_scaled = scaler.transform(df.values)
    return X_scaled, None


print('✅ Feature extraction helpers defined')

## 7 · Inference Function

Returns per-genre probabilities after aggregating all fragments at the song level.

| Ensemble | Inference strategy |
|---|---|
| **Stacking** | Fragment probs → pooled per base-model → concat → meta-LogReg |
| **WeightedVoting** | Fragment probs → pooled per base-model → weighted average |
| **Others** | Fragment probs → pooled directly |


In [ ]:
def _base_model_probs(
    name: str,
    X_scaled: np.ndarray,
    pooling: str,
) -> np.ndarray:
    """Return song-level probability vector (n_classes,) for one base model."""
    model = base_models[name]
    probs = (
        model.predict(X_scaled, verbose=0)
        if name == 'NeuralNetwork'
        else model.predict_proba(X_scaled)
    )
    return probs.max(axis=0) if pooling == 'max' else probs.mean(axis=0)


def classify(
    audio_tuple,
    model_name: str,
    pooling: str,
    confidence_threshold: float,
    progress=gr.Progress(),
) -> tuple[str, dict, str]:
    """
    Main inference function called by Gradio.
    progress is injected automatically by Gradio — do not pass it manually.
    """
    if audio_tuple is None:
        return '⚠️ No audio provided', {}, ''

    sr, audio_np = audio_tuple

    # ── Step 1: feature extraction (reports 5 % → 80 %) ─────────────────────
    progress(0.02, desc='Starting…')
    X_scaled, err = audio_to_features(
        audio_np, sr,
        progress=progress,
        progress_start=0.05,
        progress_end=0.80,
    )
    if err:
        return f'⚠️ {err}', {}, ''

    n_frags = X_scaled.shape[0]

    # ── Step 2: model inference (80 % → 95 %) ────────────────────────────────
    progress(0.80, desc=f'Running {model_name}…')

    if model_name == 'WeightedVoting':
        probs_song = sum(
            w * _base_model_probs(name, X_scaled, pooling)
            for w, name in zip(wv_weights, WV_ORDER)
        )

    elif model_name == 'Stacking':
        meta_input = np.concatenate(
            [_base_model_probs(name, X_scaled, pooling) for name in WV_ORDER]
        ).reshape(1, -1)
        probs_song = stacking_meta.predict_proba(meta_input)[0]

    else:
        probs_song = _base_model_probs(model_name, X_scaled, pooling)

    # ── Step 3: build output (95 % → 100 %) ──────────────────────────────────
    progress(0.95, desc='Preparing result…')

    probs_song = probs_song / probs_song.sum()
    probs_dict = {g: float(p) for g, p in zip(GENRES, probs_song)}
    best_genre = max(probs_dict, key=probs_dict.get)
    best_conf  = probs_dict[best_genre]

    if best_conf < confidence_threshold:
        label = (
            f'⚠️ Ambiguous — top confidence {best_conf:.1%} '
            f'is below threshold ({confidence_threshold:.0%})'
        )
    else:
        emoji_map = {'Bachata': '💃', 'Merengue': '🥁', 'Reggaeton': '🔥', 'Salsa': '🎺'}
        label = f'{emoji_map.get(best_genre, "🎵")} {best_genre}  ({best_conf:.1%})'

    details = (
        f'**Fragments analysed:** {n_frags}  ·  '
        f'**Model:** {model_name}  ·  '
        f'**Pooling:** {pooling}\n\n'
        + '  \n'.join(
            f'- **{g}**: {p:.4f}'
            for g, p in sorted(probs_dict.items(), key=lambda x: -x[1])
        )
    )

    progress(1.0, desc='Done!')
    return label, probs_dict, details


print('✅ Inference function defined')

## 8 · Gradio App

Run the cell below to launch the interactive demo.

**What to expect on Kaggle:**
- The cell finishes quickly — that is normal. Gradio v4+ runs the server in a background thread.
- The UI appears **inline** in the cell output (upload works here).
- A **public `gradio.live` link** is also printed — use that tab for **microphone recording** (mic is blocked inside the Kaggle iframe).
- The server stays alive as long as the Kaggle session is open. If you re-run the cell it cleanly restarts the server.


In [ ]:
# ── UI helpers ────────────────────────────────────────────────────────────────
MODEL_DESCRIPTIONS = {
    'LightGBM'      : 'Gradient-boosted trees — fast & interpretable',
    'XGBoost'       : 'Gradient-boosted trees — second opinion',
    'NeuralNetwork' : 'Deep MLP with BatchNorm + Dropout',
    'Stacking'      : 'LogReg meta-model trained on OOF base-model predictions',
    'WeightedVoting': 'Optuna-tuned weighted average of all base models',
}

MODEL_CHOICES = [
    (f'{name}  —  {MODEL_DESCRIPTIONS[name]}', name)
    for name in ALL_MODELS
]

INPUTS = lambda: [audio_input, model_selector, pooling_selector, confidence_slider]
OUTPUTS = lambda: [result_box, probs_chart, details_box]

# ── Build Gradio interface ────────────────────────────────────────────────────
with gr.Blocks(title='🎵 Latin Music Classifier', theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        '# 🎵 Latin Music Genre Classifier\n'
        'Classify Latin music into **Bachata · Merengue · Reggaeton · Salsa**\n\n'
        '**Upload** an audio file or **record** a short clip, then hit **Classify!**\n\n'
        '> 💡 **Tip:** for microphone recording, open the **public share link** printed '
        'below the cell — mic access is blocked inside the Kaggle iframe.'
    )

    with gr.Row():

        # ── Left column: inputs ──────────────────────────────────────────────
        with gr.Column(scale=1):

            audio_input = gr.Audio(
                sources=['upload', 'microphone'],
                type='numpy',
                label='Audio input  (upload a file  or  record from mic)',
            )

            with gr.Group():
                gr.Markdown('### ⚙️ Settings')

                model_selector = gr.Dropdown(
                    choices=MODEL_CHOICES,
                    value='WeightedVoting',
                    label='Model',
                    info='WeightedVoting is usually the strongest overall.',
                )

                pooling_selector = gr.Radio(
                    choices=['max', 'mean'],
                    value=BEST_POOLING,
                    label='Fragment pooling',
                    info=(
                        'How to aggregate probabilities across the 30 s windows. '
                        f'The training notebook found "{BEST_POOLING}" to work best.'
                    ),
                )

                confidence_slider = gr.Slider(
                    minimum=0.0,
                    maximum=1.0,
                    value=0.50,
                    step=0.05,
                    label='Confidence threshold',
                    info=(
                        'Predictions whose top probability falls below this '
                        'are flagged as ambiguous.'
                    ),
                )

            classify_btn = gr.Button('🎵  Classify!', variant='primary', size='lg')

        # ── Right column: outputs ────────────────────────────────────────────
        with gr.Column(scale=1):

            result_box = gr.Textbox(
                label='Prediction',
                interactive=False,
                lines=1,
                show_copy_button=False,
            )

            probs_chart = gr.Label(
                label='Genre probabilities',
                num_top_classes=4,
            )

            details_box = gr.Markdown(label='Details')

    # ── Wire up ───────────────────────────────────────────────────────────────
    # show_progress="full" displays the labelled progress bar with step descriptions.
    # Auto-trigger on upload is intentionally removed — CLAP takes ~5-15 s per
    # fragment; starting on upload before the user picks settings was confusing.
    classify_btn.click(
        fn=classify,
        inputs=INPUTS(),
        outputs=OUTPUTS(),
        show_progress='full',
    )

    gr.Markdown(
        '---\n'
        '**Notes**\n'
        '- Recommended input: **~30 s** of the song (chorus works best)\n'
        '- Longer clips are split into 30 s windows and each is analysed separately\n'
        '- CLAP embedding takes ~5–15 s per fragment on GPU — the progress bar shows each step\n'
        '- The classifier covers Bachata, Merengue, Reggaeton and Salsa only\n'
        '- **Microphone**: use the public share link (printed below) for full browser permissions\n'
    )


# ── Launch ────────────────────────────────────────────────────────────────────
gr.close_all()
demo.launch(share=True)